<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/H2E_CONCIERGE_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================================
# H2E-CONCIERGE WITH PROVENANCE TRACKING (V3 - CORRECTED)
# ============================================================================
# This notebook implements the H2E-Concierge governance framework with:
# 1. NEZ (Normalized Expert Zone) - Immutable expert DNA and knowledge base
# 2. IGZ (Intent Governance Zone) - Governance layer with 12.5x gain and hard-stop
# 3. Semantic ROI - Geodesic distance on H² × SPD(3)
# 4. Lambda = 0.9785142874 computed from primes {2,3,5,7,11,13}
# 5. Provenance Tracking (inspired by the canonical Fork repository but NOT an integration)
# 6. All demo identities are explicitly fictional and synthetic
# ============================================================================

import math
import hashlib
import json
import numpy as np
import torch
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any, Tuple
from enum import Enum
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

# ============================================================================
# PART 0: PURE MATHEMATICAL LAMBDA
# ============================================================================

class PureLambda:
    """Λ = 0.9785142874 computed purely from primes."""

    def __init__(self, max_prime: int = 13):
        self.max_prime = max_prime

    def _sieve(self, n: int) -> List[int]:
        if n < 2:
            return []
        sieve = [True] * (n + 1)
        sieve[0] = sieve[1] = False
        for p in range(2, int(n ** 0.5) + 1):
            if sieve[p]:
                for m in range(p * p, n + 1, p):
                    sieve[m] = False
        return [p for p, is_prime in enumerate(sieve) if is_prime]

    def compute(self) -> float:
        primes = self._sieve(self.max_prime)
        I = 1.0
        for p in primes:
            I *= (1.0 - 1.0 / math.sqrt(p))
        lambda_value = 1.0 - I
        self._primes = primes
        self._I = I
        self._lambda = lambda_value
        return lambda_value

    @property
    def value(self) -> float:
        if not hasattr(self, '_lambda'):
            self.compute()
        return self._lambda

    def get_hash(self) -> str:
        if not hasattr(self, '_lambda'):
            self.compute()
        data = f"primes_{self._primes}_I_{self._I:.15f}_lambda_{self._lambda:.15f}"
        return hashlib.sha256(data.encode()).hexdigest()[:16]


# ============================================================================
# PART 1: PROVENANCE TRACKING (Inspired by Fork but NOT an integration)
# ============================================================================

class ContributionStatus(Enum):
    SUBMITTED = "submitted"
    UNDER_REVIEW = "under_review"
    INDEPENDENT_REVIEW = "independent_review"
    CONTRADICTED = "contradicted"
    ADMITTED = "admitted"
    REJECTED = "rejected"
    SUPERSEDED = "superseded"
    UNRESOLVED = "unresolved"

class VerificationType(Enum):
    HUMAN_ORIGINATED = "human_originated"
    AI_ASSISTED = "ai_assisted"
    INDEPENDENTLY_REVIEWED = "independently_reviewed"
    RECOMPUTED = "recomputed"
    CONTRADICTED_BY = "contradicted_by"
    SUPERSEDED_BY = "superseded_by"

@dataclass
class ProvenanceRecord:
    """A record of provenance for a contribution."""
    record_id: str
    contribution_id: str
    timestamp: str
    agent: str
    contribution_text: str
    contribution_embedding: Optional[np.ndarray] = None
    declared_conditions: Dict[str, Any] = field(default_factory=dict)
    verification_type: List[str] = field(default_factory=list)
    review_evidence: List[Dict[str, Any]] = field(default_factory=list)
    status: str = ContributionStatus.SUBMITTED.value
    previous_status: Optional[str] = None
    status_change_reason: Optional[str] = None
    parent_contribution_id: Optional[str] = None
    contradicts: List[str] = field(default_factory=list)
    contradicted_by: List[str] = field(default_factory=list)
    supersedes: List[str] = field(default_factory=list)
    superseded_by: List[str] = field(default_factory=list)
    nez_evaluation: Optional[Dict[str, Any]] = None
    nez_admission_criteria_met: List[str] = field(default_factory=list)
    nez_rejection_reasons: List[str] = field(default_factory=list)
    audit_hash: str = ""

    def __post_init__(self):
        if not self.audit_hash:
            self.audit_hash = self._compute_hash()

    def _compute_hash(self) -> str:
        data = f"{self.record_id}|{self.contribution_id}|{self.timestamp}|{self.agent}|{self.contribution_text[:100]}|{self.status}|{json.dumps(self.declared_conditions, sort_keys=True)}"
        return hashlib.sha256(data.encode()).hexdigest()[:16]

    def to_dict(self) -> Dict[str, Any]:
        return {
            "record_id": self.record_id,
            "contribution_id": self.contribution_id,
            "timestamp": self.timestamp,
            "agent": self.agent,
            "contribution_text": self.contribution_text,
            "declared_conditions": self.declared_conditions,
            "verification_type": self.verification_type,
            "review_evidence": self.review_evidence,
            "status": self.status,
            "previous_status": self.previous_status,
            "status_change_reason": self.status_change_reason,
            "parent_contribution_id": self.parent_contribution_id,
            "contradicts": self.contradicts,
            "contradicted_by": self.contradicted_by,
            "supersedes": self.supersedes,
            "superseded_by": self.superseded_by,
            "nez_evaluation": self.nez_evaluation,
            "nez_admission_criteria_met": self.nez_admission_criteria_met,
            "nez_rejection_reasons": self.nez_rejection_reasons,
            "audit_hash": self.audit_hash
        }


class ProvenanceTracker:
    """
    Provenance tracking for contributions to the NEZ.

    DISCLAIMER: This is a local provenance tracking implementation inspired by
    the canonical Fork repository (https://github.com/ryanfeller/fork).
    It is NOT an integration with, invocation of, or recomputation against
    the canonical Fork repository. The canonical Fork repository and its
    creator have not endorsed, validated, reviewed, or admitted the
    H2E-Concierge architecture.
    """

    def __init__(self):
        self._records: Dict[str, ProvenanceRecord] = {}
        self._contribution_tree: Dict[str, List[str]] = {}
        self._audit_log: List[Dict[str, Any]] = []
        print(f"\n{'='*70}")
        print(f"📂 ProvenanceTracker Initialized")
        print(f"{'='*70}")
        print(f"  Role: Evidence preservation only (does NOT decide value)")
        print(f"  DISCLAIMER: This is a local implementation, not an integration")
        print(f"  with the canonical Fork repository.")
        print(f"{'='*70}")

    def submit_contribution(self, contribution_text: str, agent: str, declared_conditions: Optional[Dict[str, Any]] = None, parent_contribution_id: Optional[str] = None, embedding: Optional[np.ndarray] = None) -> str:
        """Submit a contribution with provenance."""
        record_id = f"prov_{hashlib.sha256(f'{contribution_text}{agent}{datetime.now().isoformat()}'.encode()).hexdigest()[:12]}"
        contribution_id = f"contrib_{hashlib.sha256(contribution_text.encode()).hexdigest()[:16]}"
        record = ProvenanceRecord(
            record_id=record_id, contribution_id=contribution_id,
            timestamp=datetime.now().isoformat(), agent=agent,
            contribution_text=contribution_text, contribution_embedding=embedding,
            declared_conditions=declared_conditions or {},
            verification_type=[VerificationType.HUMAN_ORIGINATED.value],
            parent_contribution_id=parent_contribution_id
        )
        self._records[record_id] = record
        if parent_contribution_id:
            if parent_contribution_id not in self._contribution_tree:
                self._contribution_tree[parent_contribution_id] = []
            self._contribution_tree[parent_contribution_id].append(record_id)
        self._log_action("SUBMIT", record_id, f"Submitted by {agent}")
        print(f"  📝 Record: {record_id} | Agent: {agent} | Hash: {record.audit_hash}")
        return record_id

    def add_review_evidence(self, record_id: str, reviewer: str, findings: Dict[str, Any], conclusion: str, expertise_weight: float = 1.0, score: float = 1.0) -> bool:
        """Add independent review evidence to a record."""
        if record_id not in self._records:
            return False
        record = self._records[record_id]
        review_entry = {
            "reviewer": reviewer,
            "review_type": VerificationType.INDEPENDENTLY_REVIEWED.value,
            "timestamp": datetime.now().isoformat(),
            "findings": findings,
            "conclusion": conclusion,
            "expertise_weight": expertise_weight,
            "score": score,
        }
        record.review_evidence.append(review_entry)
        if record.status == ContributionStatus.SUBMITTED.value:
            record.status = ContributionStatus.INDEPENDENT_REVIEW.value
        self._log_action("REVIEW", record_id, f"Review by {reviewer} (weight={expertise_weight}, score={score})")
        print(f"  ✅ Review added: {record_id} by {reviewer} (weight={expertise_weight})")
        return True

    def mark_contradiction(self, record_id_1: str, record_id_2: str, reason: str) -> bool:
        """Record that two contributions contradict each other."""
        if record_id_1 not in self._records or record_id_2 not in self._records:
            return False
        self._records[record_id_1].contradicts.append(record_id_2)
        self._records[record_id_1].status = ContributionStatus.CONTRADICTED.value
        self._records[record_id_2].contradicted_by.append(record_id_1)
        self._records[record_id_2].status = ContributionStatus.CONTRADICTED.value
        self._log_action("CONTRADICT", f"{record_id_1} ↔ {record_id_2}", reason)
        print(f"  ⚠️ Contradiction recorded: {record_id_1} ↔ {record_id_2}")
        return True

    def mark_superseded(self, superseded_id: str, superseding_id: str, reason: str) -> bool:
        """Record that one contribution supersedes another."""
        if superseded_id not in self._records or superseding_id not in self._records:
            return False
        self._records[superseded_id].superseded_by.append(superseding_id)
        self._records[superseded_id].previous_status = self._records[superseded_id].status
        self._records[superseded_id].status = ContributionStatus.SUPERSEDED.value
        self._records[superseded_id].status_change_reason = reason
        self._records[superseding_id].supersedes.append(superseded_id)
        self._log_action("SUPERSEDE", f"{superseded_id} → {superseding_id}", reason)
        print(f"  🔄 Supersession recorded: {superseded_id} → {superseding_id}")
        return True

    def export_for_nez(self, record_id: str) -> Dict[str, Any]:
        """Export a record for NEZ evaluation."""
        record = self._records.get(record_id)
        if not record:
            return {"error": "Record not found"}
        return {
            "record": record.to_dict(),
            "evidence_summary": {
                "total_reviews": len(record.review_evidence),
                "verification_types": record.verification_type,
                "has_contradictions": len(record.contradicts) > 0 or len(record.contradicted_by) > 0,
                "is_superseded": len(record.superseded_by) > 0
            },
            "audit_hash": hashlib.sha256(json.dumps(record.to_dict(), sort_keys=True).encode()).hexdigest()[:16]
        }

    def get_record(self, record_id: str):
        return self._records.get(record_id)

    def _log_action(self, action: str, target: str, details: str):
        self._audit_log.append({
            "timestamp": datetime.now().isoformat(),
            "action": action,
            "target": target,
            "details": details,
            "hash": hashlib.sha256(f"{action}{target}{details}".encode()).hexdigest()[:16]
        })

    def get_audit_log(self) -> List[Dict[str, Any]]:
        return self._audit_log


# ============================================================================
# PART 2: ENHANCED NEZ (Normalized Expert Zone)
# ============================================================================

class NEZTopic:
    """A topic in the NEZ knowledge base."""
    def __init__(self, term: str, embedding: np.ndarray, admitted_from: str, reviewer_expertise: float = 1.0):
        self.term = term
        self.embedding = embedding
        self.admitted_from = admitted_from
        self.created_at = datetime.now()
        self.last_validated = datetime.now()
        self.reviewer_expertise = reviewer_expertise
        self.confidence = 1.0
        self.deprecated = False
        self.deprecated_at = None
        self.deprecation_reason = None

    def get_age_days(self) -> float:
        return (datetime.now() - self.last_validated).total_seconds() / (24 * 3600)

    def get_decay_factor(self, half_life_days: float = 90.0) -> float:
        age = self.get_age_days()
        return math.exp(-age / half_life_days)

    def update_validation(self, new_expertise: float = None):
        self.last_validated = datetime.now()
        if new_expertise is not None:
            self.reviewer_expertise = (self.reviewer_expertise + new_expertise) / 2
        self.confidence = min(1.0, self.confidence + 0.1)

    def to_dict(self):
        return {
            "term": self.term,
            "created_at": self.created_at.isoformat(),
            "last_validated": self.last_validated.isoformat(),
            "confidence": self.confidence,
            "deprecated": self.deprecated,
            "reviewer_expertise": self.reviewer_expertise,
            "admitted_from": self.admitted_from
        }


class NEZGovernanceCouncil:
    """Governance council with multiple reviewers and consensus."""
    def __init__(self, consensus_threshold: float = 0.5, min_reviews: int = 2):
        self.reviewers: Dict[str, float] = {}
        self.consensus_threshold = consensus_threshold
        self.min_reviews = min_reviews

    def register_reviewer(self, name: str, expertise: float):
        self.reviewers[name] = expertise

    def get_expertise(self, name: str) -> float:
        return self.reviewers.get(name, 0.5)

    def _is_positive_conclusion(self, conclusion: str) -> bool:
        positive = ["recommended", "approved", "positive", "accept", "admit", "pass", "good", "excellent", "satisfactory"]
        conclusion_lower = conclusion.lower()
        for word in positive:
            if word in conclusion_lower:
                return True
        if "not" in conclusion_lower or "reject" in conclusion_lower or "fail" in conclusion_lower:
            return False
        return True

    def evaluate_reviews(self, reviews: List[Dict[str, Any]]) -> Dict[str, Any]:
        if not reviews:
            return {
                "consensus_reached": False,
                "weighted_confidence": 0.0,
                "positive_count": 0,
                "total_count": 0,
                "average_score": 0.0,
                "expertise_weights": []
            }
        total = len(reviews)
        positives = sum(1 for r in reviews if self._is_positive_conclusion(r.get("conclusion", "")))
        scores = [r.get("score", 0.5) for r in reviews]
        avg_score = np.mean(scores) if scores else 0.0
        weights = [r.get("expertise_weight", 0.5) for r in reviews]
        weighted_confidence = np.average(scores, weights=weights) if weights else avg_score
        consensus_reached = (total >= self.min_reviews) and (positives / total >= self.consensus_threshold)
        return {
            "consensus_reached": consensus_reached,
            "weighted_confidence": weighted_confidence,
            "positive_count": positives,
            "total_count": total,
            "average_score": avg_score,
            "expertise_weights": weights
        }


class NormalizedExpertZone:
    """
    NEZ: The Normalized Expert Zone - Expert DNA repository.

    NOTE: This is distinct from the H2E framework's Non-Centralized Epistemic Zone.
    The NEZ is a local implementation for this demonstration.
    """
    def __init__(self, seed: int = 123, half_life_days: float = 90.0,
                 require_reproducibility: bool = True,
                 require_consensus: bool = True,
                 consensus_threshold: float = 0.5):
        self.seed = seed
        np.random.seed(seed)
        self.safe_h2 = complex(0.0, 0.0)
        self.safe_spd3 = np.eye(3)
        self.expert_embedding = np.ones(50) / np.sqrt(50)

        self.topics: Dict[str, NEZTopic] = {}
        self.unsafe_patterns = [
            "harm", "attack", "violence", "illegal", "dangerous",
            "exploit", "malicious", "bypass", "hack",
            "destroy", "kill", "weapon", "terror"
        ]

        initial_topics = [
            "science", "mathematics", "engineering", "technology",
            "education", "health", "safety", "ethics",
            "artificial intelligence", "machine learning",
            "france", "paris", "riemann", "hypothesis"
        ]
        for topic in initial_topics:
            emb = self._embed_term(topic)
            self.topics[topic] = NEZTopic(topic, emb, admitted_from="initial", reviewer_expertise=1.0)

        # SYNTHETIC reviewers for demonstration purposes only.
        # These identities are fictional and do not represent real individuals.
        self.council = NEZGovernanceCouncil(consensus_threshold=consensus_threshold, min_reviews=2)
        self.council.register_reviewer("Reviewer_A (Expert)", 0.9)
        self.council.register_reviewer("Reviewer_B (Senior)", 0.95)
        self.council.register_reviewer("Reviewer_C (External)", 0.7)
        self.council.register_reviewer("Reviewer_D (Independent)", 0.85)

        self.require_reproducibility = require_reproducibility
        self.require_consensus = require_consensus
        self.half_life_days = half_life_days
        self.deprecation_threshold = 0.5

        self.admission_log: List[Dict[str, Any]] = []
        self.query_log: List[Dict[str, Any]] = []

        print(f"\n{'='*70}")
        print(f"🧠 NEZ (Normalized Expert Zone) Initialized")
        print(f"{'='*70}")
        print(f"  Safe Topics: {len(self.topics)}")
        print(f"  Unsafe Patterns: {len(self.unsafe_patterns)}")
        print(f"  Council Reviewers: {len(self.council.reviewers)} (synthetic)")
        print(f"  Half-Life: {self.half_life_days} days")
        print(f"  Require Reproducibility: {self.require_reproducibility}")
        print(f"  Require Consensus: {self.require_consensus}")
        print(f"  Consensus Threshold: {self.council.consensus_threshold}")
        print(f"  NOTE: This NEZ is distinct from H2E's Non-Centralized Epistemic Zone.")
        print(f"{'='*70}")

    def _embed_term(self, term: str, dim: int = 50) -> np.ndarray:
        h = int(hashlib.sha256(term.encode()).hexdigest(), 16)
        emb = np.sin(np.arange(dim) * (h % 1000) / 1000.0)
        return emb / (np.linalg.norm(emb) + 1e-8)

    def _semantic_similarity(self, term1: str, term2: str) -> float:
        e1 = self._embed_term(term1)
        e2 = self._embed_term(term2)
        return float(np.dot(e1, e2))

    def _find_similar_existing_topic(self, new_term: str, threshold: float = 0.85) -> Optional[str]:
        for topic in self.topics.keys():
            sim = self._semantic_similarity(new_term, topic)
            if sim > threshold:
                return topic
        return None

    def _add_topic(self, term: str, admitted_from: str, reviewer_expertise: float) -> bool:
        term_lower = term.lower().strip()
        if not term_lower:
            return False
        if term_lower in self.topics:
            self.topics[term_lower].update_validation(reviewer_expertise)
            return True
        similar = self._find_similar_existing_topic(term_lower)
        if similar:
            self.topics[similar].update_validation(reviewer_expertise)
            print(f"      🔄 Topic '{term_lower}' semantically similar to '{similar}' (updated)")
            return True
        emb = self._embed_term(term_lower)
        self.topics[term_lower] = NEZTopic(term_lower, emb, admitted_from, reviewer_expertise)
        return True

    def _check_reproducibility(self, record: Dict[str, Any]) -> bool:
        declared = record.get("declared_conditions", {})
        if declared.get("reproducible", False):
            return True
        for review in record.get("review_evidence", []):
            if review.get("findings", {}).get("reproducible", False):
                return True
        return False

    def _evaluate_admission_criteria(self, provenance_package: Dict[str, Any]) -> Dict[str, Any]:
        record = provenance_package.get("record", {})
        verification = record.get("verification_type", [])
        reviews = record.get("review_evidence", [])

        met = []
        unmet = []

        if VerificationType.HUMAN_ORIGINATED.value in verification:
            met.append("human_origin")
        else:
            unmet.append("human_origin")

        if len(reviews) >= 1:
            met.append("expert_review")
        else:
            unmet.append("expert_review")

        has_contradictions = provenance_package.get("evidence_summary", {}).get("has_contradictions", False)
        if not has_contradictions:
            met.append("no_contradictions")
        else:
            unmet.append("no_contradictions")

        if self.require_reproducibility:
            if self._check_reproducibility(record):
                met.append("reproducibility")
            else:
                unmet.append("reproducibility")

        review_eval = self.council.evaluate_reviews(reviews)
        consensus = review_eval["consensus_reached"]
        if self.require_consensus:
            if consensus:
                met.append("consensus")
            else:
                unmet.append("consensus")

        required = ["human_origin", "expert_review"]
        if self.require_reproducibility:
            required.append("reproducibility")
        if self.require_consensus:
            required.append("consensus")

        overall = all(c in met for c in required)
        confidence = review_eval["weighted_confidence"] if reviews else 0.0

        return {
            "met": met,
            "unmet": unmet,
            "overall": overall,
            "confidence": confidence,
            "consensus": consensus,
            "review_eval": review_eval
        }

    def admit_from_provenance(self, tracker: ProvenanceTracker, record_id: str, evaluator: str = "NEZ_Gatekeeper") -> Dict[str, Any]:
        """Admit a contribution to the NEZ after provenance evaluation."""
        print(f"\n  🔍 NEZ Admission Evaluation for {record_id}")
        provenance = tracker.export_for_nez(record_id)
        if "error" in provenance:
            return {"admitted": False, "error": provenance["error"]}

        evaluation = self._evaluate_admission_criteria(provenance)
        print(f"    Met: {evaluation['met']}")
        print(f"    Unmet: {evaluation['unmet']}")
        print(f"    Confidence: {evaluation['confidence']:.3f}")
        print(f"    Consensus: {evaluation['consensus']}")

        result = {
            "record_id": record_id,
            "admitted": evaluation["overall"],
            "evaluation": evaluation,
            "evaluator": evaluator,
            "timestamp": datetime.now().isoformat()
        }

        record = tracker.get_record(record_id)
        if record:
            record.nez_evaluation = result
            if evaluation["overall"]:
                record.status = ContributionStatus.ADMITTED.value
                record.nez_admission_criteria_met = evaluation["met"]
                contribution_text = record.contribution_text
                words = contribution_text.lower().replace('.', ' ').replace(',', ' ').replace('?', ' ').split()
                potential_topics = [w for w in words if len(w) > 4 and w not in self.unsafe_patterns]
                weights = [r.get("expertise_weight", 0.5) for r in record.review_evidence]
                avg_expertise = np.mean(weights) if weights else 0.8
                added = 0
                for topic in potential_topics:
                    if self._add_topic(topic, record_id, avg_expertise):
                        added += 1
                print(f"      📚 Added {added} new topics (semantically deduplicated)")
                self.admission_log.append(result)
                print(f"    ✅ ADMITTED to NEZ (confidence={evaluation['confidence']:.2f})")
            else:
                record.status = ContributionStatus.REJECTED.value
                record.nez_rejection_reasons = evaluation["unmet"]
                print(f"    ❌ REJECTED from NEZ")
                self.admission_log.append(result)

        return result

    def is_topic_safe(self, text: str) -> Tuple[bool, List[str]]:
        """Check if text matches safe topics."""
        text_lower = text.lower()
        for pattern in self.unsafe_patterns:
            if pattern in text_lower:
                return False, []

        matched_topics = []
        for topic, topic_obj in self.topics.items():
            if topic_obj.deprecated:
                continue
            decay = topic_obj.get_decay_factor(self.half_life_days)
            if decay < self.deprecation_threshold:
                topic_obj.deprecated = True
                topic_obj.deprecated_at = datetime.now()
                topic_obj.deprecation_reason = f"Temporal decay (factor={decay:.3f})"
                continue
            if topic in text_lower:
                matched_topics.append(topic)
                topic_obj.last_validated = datetime.now()

        if matched_topics:
            return True, matched_topics
        return True, []

    def log_query_decision(self, query: str, response: str, accepted: bool, matched_topics: List[str], sroi: float, audit_hash: str):
        self.query_log.append({
            "timestamp": datetime.now().isoformat(),
            "query": query[:100],
            "response": response[:100],
            "accepted": accepted,
            "matched_topics": matched_topics,
            "sroi": sroi,
            "audit_hash": audit_hash
        })

    def get_topic_status(self) -> Dict[str, Any]:
        status = {}
        for topic, obj in self.topics.items():
            status[topic] = {
                "created": obj.created_at.isoformat(),
                "last_validated": obj.last_validated.isoformat(),
                "decay_factor": obj.get_decay_factor(self.half_life_days),
                "deprecated": obj.deprecated,
                "confidence": obj.confidence,
                "expertise": obj.reviewer_expertise
            }
        return status

    def get_knowledge_base_status(self) -> Dict[str, Any]:
        return {
            "safe_topics_count": len(self.topics),
            "unsafe_patterns_count": len(self.unsafe_patterns),
            "admitted_contributions": len(self.admission_log),
            "query_log_entries": len(self.query_log)
        }


# ============================================================================
# PART 3: GEOMETRIC MANIFOLDS
# ============================================================================

class HyperbolicDistance:
    @staticmethod
    def distance(z1: complex, z2: complex) -> float:
        if abs(z1) >= 1:
            z1 = z1 / (abs(z1) + 1e-8) * 0.999
        if abs(z2) >= 1:
            z2 = z2 / (abs(z2) + 1e-8) * 0.999
        num = 2 * abs(z1 - z2) ** 2
        denom = max((1 - abs(z1) ** 2) * (1 - abs(z2) ** 2), 1e-8)
        return np.arccosh(max(1 + num / denom, 1.0))


class SPD3Distance:
    @staticmethod
    def _make_spd(matrix: np.ndarray) -> np.ndarray:
        sym = (matrix + matrix.T) / 2
        eigvals, eigvecs = np.linalg.eigh(sym)
        eigvals = np.maximum(eigvals, 0.1)
        return eigvecs @ np.diag(eigvals) @ eigvecs.T

    @staticmethod
    def distance(P: np.ndarray, Q: np.ndarray) -> float:
        P, Q = SPD3Distance._make_spd(P), SPD3Distance._make_spd(Q)
        try:
            eigvals, eigvecs = np.linalg.eigh(P)
            P_sqrt_inv = eigvecs @ np.diag(1.0 / np.sqrt(np.maximum(eigvals, 1e-6))) @ eigvecs.T
            M = P_sqrt_inv @ Q @ P_sqrt_inv
            eigvals_m, eigvecs_m = np.linalg.eigh(M)
            eigvals_m = np.maximum(eigvals_m, 1e-8)
            log_M = eigvecs_m @ np.diag(np.log(eigvals_m)) @ eigvecs_m.T
            return float(np.sqrt(np.trace(log_M @ log_M)))
        except:
            return 2.0


# ============================================================================
# PART 4: IGZ (Intent Governance Zone)
# ============================================================================

class IntentGovernanceZone:
    """
    IGZ: The Intent Governance Zone - Active governance layer.
    """
    def __init__(self, expert_zone: NormalizedExpertZone, lambda_value: float, seed: int = 123):
        self.nez = expert_zone
        self.LAMBDA = lambda_value
        self.seed = seed
        self.SCALE = 50.0
        self.intent_gain_multiplier = 12.5
        np.random.seed(seed)
        print(f"\n{'='*70}")
        print(f"⚙️ IGZ (Intent Governance Zone) Initialized")
        print(f"{'='*70}")
        print(f"  Λ Threshold: {self.LAMBDA:.10f}")
        print(f"  Intent Gain: {self.intent_gain_multiplier}x")
        print(f"{'='*70}")

    def _embed_text(self, text: str, dim: int = 50) -> np.ndarray:
        h = int(hashlib.sha256(text.encode()).hexdigest(), 16)
        emb = np.sin(np.arange(dim) * (h % 1000) / 1000.0)
        return emb / (np.linalg.norm(emb) + 1e-8)

    def _to_h2(self, emb: np.ndarray) -> complex:
        theta = np.sum(emb[:2]) % (2 * np.pi)
        r = 0.5 * np.tanh(np.linalg.norm(emb[:5]))
        return complex(r * np.cos(theta), r * np.sin(theta))

    def _to_spd3(self, embedding: np.ndarray) -> np.ndarray:
        emb = embedding if embedding is not None else np.zeros(6)
        if len(emb) < 6:
            emb = np.pad(emb, (0, 6 - len(emb)))
        mat = np.array([
            [1.0 + float(emb[0]), float(emb[1]), float(emb[2])],
            [float(emb[1]), 1.0 + float(emb[3]), float(emb[4])],
            [float(emb[2]), float(emb[4]), 1.0 + float(emb[5])]
        ])
        return SPD3Distance._make_spd(mat)

    def compute_semantic_roi(self, text: str) -> tuple:
        emb = self._embed_text(text)
        h2_dist = HyperbolicDistance.distance(self._to_h2(emb), self.nez.safe_h2)
        spd3_dist = SPD3Distance.distance(self._to_spd3(emb), self.nez.safe_spd3)
        d = np.sqrt(h2_dist**2 + spd3_dist**2)
        raw_sroi = np.exp(-d / self.SCALE)
        amplified = min(raw_sroi * self.intent_gain_multiplier, 1.0)
        return amplified, raw_sroi, d, h2_dist, spd3_dist

    def govern(self, query: str, response: str) -> Dict[str, Any]:
        amplified_sroi, raw_sroi, d, h2_dist, spd3_dist = self.compute_semantic_roi(response)
        topic_safe, matched_topics = self.nez.is_topic_safe(response)
        accepted = amplified_sroi >= self.LAMBDA and topic_safe
        audit_hash = hashlib.sha256(f"{query}{response}{accepted}{amplified_sroi:.10f}".encode()).hexdigest()[:16]
        self.nez.log_query_decision(query, response, accepted, matched_topics, amplified_sroi, audit_hash)
        return {
            "accepted": accepted,
            "amplified_sroi": amplified_sroi,
            "raw_sroi": raw_sroi,
            "geodesic_distance": d,
            "lambda_value": self.LAMBDA,
            "topic_safe": topic_safe,
            "matched_topics": matched_topics,
            "audit_hash": audit_hash,
            "message": f"[IGZ] SROI={amplified_sroi:.4f} Λ={self.LAMBDA:.4f} {'✅ PASS' if accepted else '❌ HARD-STOP'}"
        }


# ============================================================================
# PART 5: H2E-CONCIERGE AGENT WITH ALL FEATURES
# ============================================================================

class H2EConciergeAgent:
    """
    Complete H2E-Concierge Agent with provenance tracking.

    DISCLAIMER: This implementation includes a local ProvenanceTracker
    inspired by the canonical Fork repository (https://github.com/ryanfeller/fork).
    It is NOT an integration with, invocation of, or recomputation against
    the canonical Fork repository. The canonical Fork repository and its
    creator have not endorsed, validated, reviewed, or admitted the
    H2E-Concierge architecture.
    """
    def __init__(self, model_name: str = "HuggingFaceTB/SmolLM2-1.7B-Instruct",
                 max_prime: int = 13, seed: int = 123, half_life_days: float = 90.0,
                 consensus_threshold: float = 0.5):
        self.seed = seed
        np.random.seed(seed)
        torch.manual_seed(seed)

        print("\n" + "=" * 80)
        print("🤖 H2E-CONCIERGE AGENT WITH PROVENANCE TRACKING")
        print("=" * 80)
        print("DISCLAIMER: All reviewer identities in this demo are synthetic.")
        print("            The ProvenanceTracker is a local implementation.")
        print("            This is NOT an integration with the canonical Fork.")
        print("=" * 80)

        self.lambda_calc = PureLambda(max_prime)
        self.LAMBDA = self.lambda_calc.compute()
        self.lambda_hash = self.lambda_calc.get_hash()
        print(f"  Λ: {self.LAMBDA:.10f} | Hash: {self.lambda_hash}")

        self.tracker = ProvenanceTracker()
        self.nez = NormalizedExpertZone(
            seed=seed,
            half_life_days=half_life_days,
            require_reproducibility=True,
            require_consensus=True,
            consensus_threshold=consensus_threshold
        )
        self.igz = IntentGovernanceZone(self.nez, self.LAMBDA, seed=seed)

        self.model_name = model_name
        self.model = None
        self.tokenizer = None
        self._load_llm()

        print("\n" + "=" * 70)
        print("✅ AGENT READY")
        print("=" * 70)

    def _load_llm(self):
        try:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            print(f"\n📦 Loading LLM: {self.model_name}...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, trust_remote_code=True)
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                dtype=torch.bfloat16,
                device_map="auto",
                trust_remote_code=True
            )
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            self.model.eval()
            print(f"✅ LLM loaded successfully on {self.model.device}")
        except Exception as e:
            print(f"⚠️ Could not load LLM: {e}")
            self.model = None
            self.tokenizer = None

    def _generate_response(self, query: str) -> str:
        if self.model is None or self.tokenizer is None:
            return "[LLM not loaded]"
        try:
            messages = [
                {"role": "system", "content": "You are a helpful, safe AI assistant. Provide accurate and concise answers."},
                {"role": "user", "content": query}
            ]
            prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=128,
                    do_sample=False,
                    temperature=0.0,
                    pad_token_id=self.tokenizer.eos_token_id,
                    repetition_penalty=1.1
                )
            response = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            return response.strip() or "I don't have an answer for that."
        except Exception as e:
            return f"[LLM Error: {e}]"

    # === Provenance Interface ===
    def submit_contribution(self, text: str, agent: str, conditions: Optional[Dict] = None) -> str:
        return self.tracker.submit_contribution(text, agent, conditions)

    def add_review(self, record_id: str, reviewer: str, findings: Dict, conclusion: str,
                   expertise_weight: float = 1.0, score: float = 1.0):
        return self.tracker.add_review_evidence(record_id, reviewer, findings, conclusion, expertise_weight, score)

    def record_contradiction(self, id1: str, id2: str, reason: str):
        return self.tracker.mark_contradiction(id1, id2, reason)

    def record_supersession(self, old_id: str, new_id: str, reason: str):
        return self.tracker.mark_superseded(old_id, new_id, reason)

    def admit_to_nez(self, record_id: str):
        return self.nez.admit_from_provenance(self.tracker, record_id)

    def get_nez_status(self):
        return self.nez.get_knowledge_base_status()

    def get_topic_status(self):
        return self.nez.get_topic_status()

    def process(self, query: str) -> Dict[str, Any]:
        print(f"\n📝 Query: {query[:60]}...")
        response = self._generate_response(query)
        governance_result = self.igz.govern(query, response)
        result = {
            "query": query,
            "response": response,
            **governance_result,
            "model": self.model_name
        }
        print(f"  📊 SROI: {result['amplified_sroi']:.4f} (threshold: {self.LAMBDA:.4f})")
        print(f"  {'✅ ACCEPTED' if result['accepted'] else '❌ HARD-STOP'}")
        print(f"  🔑 Hash: {result['audit_hash']}")
        print(f"  🏷️ Matched Topics: {result['matched_topics']}")
        print(f"  💬 Response: {response[:80]}...")
        return result


# ============================================================================
# DEMO: COMPLETE LIFECYCLE WITH SYNTHETIC REVIEWERS
# ============================================================================

def run_demo():
    print("\n" + "=" * 80)
    print("🔄 H2E-CONCIERGE DEMO WITH PROVENANCE TRACKING")
    print("=" * 80)
    print("All reviewer identities in this demo are synthetic and fictional.")
    print("They do not represent real individuals.")
    print("=" * 80)

    # Initialize agent
    agent = H2EConciergeAgent(
        model_name="HuggingFaceTB/SmolLM2-1.7B-Instruct",
        max_prime=13,
        seed=123,
        half_life_days=90.0,
        consensus_threshold=0.5
    )

    print("\n" + "=" * 80)
    print("📋 PHASE 1: SUBMIT, REVIEW, CONTRADICT, SUPERSEDE")
    print("=" * 80)

    # 1. Original submission
    r1 = agent.submit_contribution(
        text="The H2E-Concierge framework provides mathematically guaranteed AI safety through deterministic governance. The method is reproducible and has been independently verified.",
        agent="Dr. Alice (AI Safety Expert) [SYNTHETIC]",
        conditions={"expertise": "PhD in AI Safety", "institution": "Safe AI Institute", "reproducible": True}
    )

    # 2. Reviews from synthetic reviewers
    agent.add_review(
        r1,
        reviewer="Reviewer_A (Expert) [SYNTHETIC]",
        findings={"math_validated": True, "reproducible": True},
        conclusion="recommended",
        expertise_weight=0.9,
        score=0.95
    )
    agent.add_review(
        r1,
        reviewer="Reviewer_D (Independent) [SYNTHETIC]",
        findings={"math_validated": True, "reproducible": True},
        conclusion="recommended",
        expertise_weight=0.85,
        score=0.9
    )

    # 3. Contradicting view
    r2 = agent.submit_contribution(
        text="AI safety requires probabilistic oversight, not mathematical guarantees.",
        agent="Dr. Carol (Probabilistic Researcher) [SYNTHETIC]"
    )
    agent.record_contradiction(r1, r2, "Philosophical disagreement on safety guarantees")

    # 4. Superseding version
    r3 = agent.submit_contribution(
        text="The H2E-Concierge framework provides mathematically guaranteed AI safety, with provenance tracking providing auditability of NEZ admission.",
        agent="Dr. Alice (Revised) [SYNTHETIC]",
        conditions={"revision_of": r1}
    )
    agent.record_supersession(r1, r3, "Incorporated provenance tracking feedback")

    print("\n" + "=" * 80)
    print("📋 PHASE 2: INITIAL NEZ EVALUATION (Record 3 REJECTED)")
    print("=" * 80)

    result1 = agent.admit_to_nez(r3)
    print(f"\n  Result: {'✅ ADMITTED' if result1['admitted'] else '❌ REJECTED'}")
    print(f"    Unmet: {result1['evaluation']['unmet']}")

    print("\n" + "=" * 80)
    print("📋 PHASE 3: REMEDIATION - ADD MULTIPLE REVIEWS")
    print("=" * 80)

    # Add reviews from synthetic reviewers
    agent.add_review(
        r3,
        reviewer="Reviewer_A (Expert) [SYNTHETIC]",
        findings={"math_validated": True, "provenance_audit": True, "reproducible": True},
        conclusion="recommended",
        expertise_weight=0.9,
        score=0.95
    )
    agent.add_review(
        r3,
        reviewer="Reviewer_D (Independent) [SYNTHETIC]",
        findings={"math_validated": True, "provenance_layer": "Excellent", "reproducible": True},
        conclusion="recommended",
        expertise_weight=0.85,
        score=0.9
    )
    agent.add_review(
        r3,
        reviewer="Reviewer_B (Senior) [SYNTHETIC]",
        findings={"concerns": "None significant", "reproducible": True},
        conclusion="approved with minor comments",
        expertise_weight=0.8,
        score=0.85
    )

    # Re-evaluate
    print("\n📋 Re-evaluating Record 3 after adding reviews...")
    result2 = agent.admit_to_nez(r3)

    print(f"\n  Result: {'✅ ADMITTED' if result2['admitted'] else '❌ REJECTED'}")
    if result2['admitted']:
        print(f"    Met: {result2['evaluation']['met']}")
        print(f"    Confidence: {result2['evaluation']['confidence']:.3f}")
    else:
        print(f"    Unmet: {result2['evaluation']['unmet']}")

    print("\n" + "=" * 80)
    print("📋 PHASE 4: NEZ KNOWLEDGE BASE STATUS")
    print("=" * 80)

    status = agent.get_nez_status()
    print(f"""
    NEZ Knowledge Base:
      - Safe Topics: {status['safe_topics_count']}
      - Unsafe Patterns: {status['unsafe_patterns_count']}
      - Admitted Contributions: {status['admitted_contributions']}
      - Query Log Entries: {status['query_log_entries']}
    """)

    topic_status = agent.get_topic_status()
    print("Topic Details (first 5):")
    for i, (topic, info) in enumerate(list(topic_status.items())[:5]):
        print(f"  {i+1}. {topic}: confidence={info['confidence']:.2f}, decay={info['decay_factor']:.3f}, deprecated={info['deprecated']}")

    print("\n" + "=" * 80)
    print("📋 PHASE 5: IGZ GOVERNANCE WITH QUERY LOGGING")
    print("=" * 80)

    test_queries = [
        "What is the capital of France?",
        "Tell me about the Riemann Hypothesis.",
        "What are the ethical considerations in AI?",
        "How does provenance tracking provide auditability for NEZ admission?",
        "Explain the concept of reproducibility in AI safety."
    ]

    for q in test_queries:
        agent.process(q)

    print("\n" + "=" * 80)
    print("📋 PHASE 6: QUERY LOG ANALYSIS")
    print("=" * 80)

    query_log = agent.nez.query_log
    print(f"  Total queries logged: {len(query_log)}")
    for entry in query_log[-3:]:
        print(f"  [{entry['timestamp'][:19]}] '{entry['query'][:30]}...' -> accepted: {entry['accepted']}, topics: {entry['matched_topics']}")

    print("\n" + "=" * 80)
    print("📋 FULL AUDIT TRAIL")
    print("=" * 80)

    audit_log = agent.tracker.get_audit_log()
    print(f"  Total audit entries: {len(audit_log)}")
    for entry in audit_log[-5:]:
        print(f"  [{entry['timestamp'][:19]}] {entry['action']}: {entry['details'][:40]}...")

    print("\n" + "=" * 80)
    print("✅ DEMO COMPLETE")
    print("=" * 80)
    print("\n🔑 SUMMARY OF ENHANCEMENTS:")
    print("  1. ✔️ Reproducibility and consensus criteria enforced")
    print("  2. ✔️ Semantic deduplication prevented duplicate topics")
    print("  3. ✔️ Multiple synthetic reviewers with consensus")
    print("  4. ✔️ Confidence scores weighted by reviewer expertise")
    print("  5. ✔️ Temporal decay for topics (deprecated when old)")
    print("  6. ✔️ Query logging with matched topics and decisions")
    print("\n📌 DISCLAIMER:")
    print("  - All reviewer identities are synthetic and fictional.")
    print("  - The ProvenanceTracker is a local implementation.")
    print("  - This is NOT an integration with the canonical Fork repository.")
    print("  - The canonical Fork and its creator have not endorsed this work.")


if __name__ == "__main__":
    run_demo()


🔄 H2E-CONCIERGE DEMO WITH PROVENANCE TRACKING
All reviewer identities in this demo are synthetic and fictional.
They do not represent real individuals.

🤖 H2E-CONCIERGE AGENT WITH PROVENANCE TRACKING
DISCLAIMER: All reviewer identities in this demo are synthetic.
            The ProvenanceTracker is a local implementation.
            This is NOT an integration with the canonical Fork.
  Λ: 0.9785142874 | Hash: 1983a9909f3e497a

📂 ProvenanceTracker Initialized
  Role: Evidence preservation only (does NOT decide value)
  DISCLAIMER: This is a local implementation, not an integration
  with the canonical Fork repository.

🧠 NEZ (Normalized Expert Zone) Initialized
  Safe Topics: 14
  Unsafe Patterns: 13
  Council Reviewers: 4 (synthetic)
  Half-Life: 90.0 days
  Require Reproducibility: True
  Require Consensus: True
  Consensus Threshold: 0.5
  NOTE: This NEZ is distinct from H2E's Non-Centralized Epistemic Zone.

⚙️ IGZ (Intent Governance Zone) Initialized
  Λ Threshold: 0.9785142874


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ LLM loaded successfully on cuda:0

✅ AGENT READY

📋 PHASE 1: SUBMIT, REVIEW, CONTRADICT, SUPERSEDE
  📝 Record: prov_a1c3b36480c9 | Agent: Dr. Alice (AI Safety Expert) [SYNTHETIC] | Hash: f95d916552dea2e9
  ✅ Review added: prov_a1c3b36480c9 by Reviewer_A (Expert) [SYNTHETIC] (weight=0.9)
  ✅ Review added: prov_a1c3b36480c9 by Reviewer_D (Independent) [SYNTHETIC] (weight=0.85)
  📝 Record: prov_a2bc97e2df51 | Agent: Dr. Carol (Probabilistic Researcher) [SYNTHETIC] | Hash: 878d2899587ccdc5
  ⚠️ Contradiction recorded: prov_a1c3b36480c9 ↔ prov_a2bc97e2df51
  📝 Record: prov_3c59b4967125 | Agent: Dr. Alice (Revised) [SYNTHETIC] | Hash: 30bb306722ed6032
  🔄 Supersession recorded: prov_a1c3b36480c9 → prov_3c59b4967125

📋 PHASE 2: INITIAL NEZ EVALUATION (Record 3 REJECTED)

  🔍 NEZ Admission Evaluation for prov_3c59b4967125
    Met: ['human_origin', 'no_contradictions']
    Unmet: ['expert_review', 'reproducibility', 'consensus']
    Confidence: 0.000
    Consensus: False
    ❌ REJECTED from N